# RL with custom rewards
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/v1/cookbook/notebooks/rl_with_custom_rewards.ipynb)

Define a charged-residue reward, run a short GRPO job, and compare scores before and after training.


## Setup
A GPU runtime is recommended: **Runtime → Change runtime type → GPU**. CPU execution is supported but slower.

Run cells in order. Installation is self-contained; no repository clone or account is needed. If Colab requests a session restart after installation, restart before running the imports. The first model load downloads weights.


In [ ]:
import sys

!"{sys.executable}" -m pip install -q "idiom[cookbook] @ git+https://github.com/rotskoff-group/idiom.git@v1"

In [1]:
import gc
import time
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

from idiom import IDiom
from idiom.data.records import Record
from idiom.utils.notebook_helpers import save_run, train_and_save, training_config, write_fasta

started = time.perf_counter()

## Settings


In [2]:
MODEL_ID = "jxliu2/idiom-20M" # Pretrained IDiom model or local release directory
DEVICE = "auto" # "cpu" or "cuda"; "auto" uses an available GPU
BATCH_SIZE = 1 # Prompts per training batch; sequences per inference batch
SEED = 0 # Random seed
MAX_STEPS = 10 # Training steps for this demonstration
TARGET_LENGTH = 50 # Target IDR length in residues
MAX_NEW_TOKENS = 96 # Generation limit, including STOP; not a fixed IDR length
N = 8 # Sequences sampled from each model for comparison
OUT_DIR = Path("reward_design_outputs") / time.strftime("%Y%m%d-%H%M%S") # New timestamped folder per run
TARGET_CHARGED_FRACTION = 0.3 # Target fraction of D, E, K, and R residues

In [3]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Define the reward
The factory returns one finite score per sequence, including empty sequences. Edit this scorer and its target together. Length and entropy terms below discourage simple shortcuts.


In [4]:
from idiom.train.grpo.reward.resolve import load_callable

reward_file = OUT_DIR / "custom_reward.py"
reward_file.write_text(
    "def charged_fraction():\n"
    "    def score(sequences):\n"
    '        return [sum(s.count(a) for a in "DEKR") / max(len(s), 1) for s in sequences]\n'
    "    return score\n"
)
reward_spec = dict(name=f"{reward_file.resolve()}:charged_fraction")

scorer = load_callable(reward_spec["name"])()
assert scorer(["", "DEKR", "AAAA"]) == [0.0, 1.0, 0.0]

## Sample the starting model


In [5]:
base = IDiom.from_pretrained(MODEL_ID, device=DEVICE)
sampling = dict(n=N, batch_size=BATCH_SIZE, seed=SEED, temperature=1.0, max_new_tokens=MAX_NEW_TOKENS)
baseline = base.generate_unprompted(**sampling)
write_fasta(
    [Record(f"baseline_{i}", s, 0, len(s)) for i, s in enumerate(baseline) if s],
    OUT_DIR / "baseline.fasta",
)
del base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Train
Run a small demonstration job. Training saves a configuration, CSV metrics, a checkpoint, and a reloadable model in `OUT_DIR`. Logging stays local; no account or API key is required.


In [6]:
cfg = training_config("grpo", MODEL_ID, OUT_DIR, device=DEVICE, steps=MAX_STEPS, seed=SEED)
cfg.prompts.n = max(16, MAX_STEPS * BATCH_SIZE)
cfg.prompts.batch_size = BATCH_SIZE
cfg.grpo.group_size = 2
cfg.grpo.max_new_tokens = MAX_NEW_TOKENS
cfg.grpo.lr = 5e-6
cfg.grpo.track_disorder = False
cfg.grpo.log_samples_every = 0
terms = [
    dict(
        label="length",
        weight=1.0,
        reward=dict(name="length"),
        shaping=dict(name="quadratic", target=TARGET_LENGTH, width=0.2),
    ),
    dict(
        label="entropy",
        weight=1.0,
        reward=dict(name="entropy"),
        shaping=dict(name="quadratic", target=3.65, width=0.2),
    ),
]

terms.append(
    dict(
        label="charged_fraction",
        weight=1.0,
        reward=reward_spec,
        shaping=dict(name="quadratic", target=TARGET_CHARGED_FRACTION, width=0.2),
    )
)

cfg.reward.terms = terms
release = train_and_save(cfg)

## Inspect training
Inspect the latest logged metrics. Full history is in `training/metrics/`. Blank entries indicate a metric was not logged on that step.


In [7]:
metrics_path = next((OUT_DIR / "training/metrics").glob("version_*/metrics.csv"))
metrics = pd.read_csv(metrics_path)
columns = ["step"] + [name for name in ["train/loss", "train/reward", "train/kl"] if name in metrics]
print(f"Logged {len(metrics)} metric rows; latest values:")
display(metrics[columns].tail(3))

Logged 10 metric rows; latest values:


,step,train/loss,train/reward,train/kl
7,7,0.149702,-16.829266,0.000053
8,8,0.118882,-8.668839,0.000059
9,9,0.238710,-14.344069,0.000028


## Generate from the saved model
Use the same sampling settings as the baseline for the comparison.


In [8]:
adapted_model = IDiom.from_pretrained(release, device=DEVICE)
adapted = adapted_model.generate_unprompted(**sampling)
write_fasta(
    [Record(f"adapted_{i}", s, 0, len(s)) for i, s in enumerate(adapted) if s],
    OUT_DIR / "adapted.fasta",
)
del adapted_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Inspect scores
Compare raw scores and the total shaped reward. Short demonstration runs may not improve the objective and do not establish sequence function. Full per-sequence scores are saved in `rewards.csv`.


In [9]:
from idiom.train.grpo.reward import build_reward

objective = build_reward(cfg.reward)
rows = []
for group, sequences in (("baseline", baseline), ("adapted", adapted)):
    totals, details = objective(sequences, group_size=1)
    scores = pd.DataFrame(details)
    scores["total_reward"] = totals
    scores["sequence"] = sequences
    scores["group"] = group
    rows.append(scores)
comparison = pd.concat(rows, ignore_index=True)
comparison.to_csv(OUT_DIR / "rewards.csv", index=False)
with pd.option_context("display.max_colwidth", 80):
    display(comparison[["group", "sequence", "total_reward"]].groupby("group", sort=False).head(2))

summary = comparison.groupby("group", sort=False).mean(numeric_only=True)
summary.to_csv(OUT_DIR / "reward_summary.csv")
score_columns = [f"{term['label']}_raw" for term in terms] + ["total_reward"]
display(summary[score_columns].round(3))

,group,sequence,total_reward
0,baseline,DKKEWGNEEEHEDSGGRPHAPPQASSIQVL,-4.315643
1,baseline,SSRKKRRKPAAADPPPYDSSDSDAVLGPAGDFSALSLPDEPVHPAAPSYSPAESSDSDVQLVSPPDDDYHHALAAR...,-22.276915
8,adapted,DKKEWGNEAEHKASGGRPHAPPQASSIQVL,-4.316427
9,adapted,SSRKKRRKPAAADPPPYDSSDSSAVLGPAGDFSALSLPDEPVHPRRTSYRPAESSDSDEQLVSVPDDDYEDAEAAR...,-21.234311


,length_raw,entropy_raw,charged_fraction_raw,total_reward
group,,,,
baseline,51.625,3.586,0.235,-5.973
adapted,51.750,3.595,0.238,-5.856


## Results
`baseline.fasta`, `adapted.fasta`, and `rewards.csv` contain the before/after samples and scores. `custom_reward.py` saves your scorer. `model/` is a reloadable release; `training/` contains logs and a checkpoint.

`run.json` records the settings and package versions. Open the output folder in Colab’s **Files** pane to download results. Download them before the runtime ends, or copy them to mounted Drive. Saved notebook previews do not include the exported files.


Saved previews show a CPU example run. Run the cells to create the exported files.

In [10]:
save_run(
    OUT_DIR,
    dict(model=MODEL_ID, device=DEVICE, seed=SEED, steps=MAX_STEPS, sampling=sampling),
    elapsed=time.perf_counter() - started,
)
print("Results folder:", OUT_DIR)

Results folder: reward_design_outputs/20260921-134305
